# SQLSage Training Runbook (Colab T4-first)

This notebook is updated for a **T4-first** workflow with your deployed Space as the live environment endpoint.

## Before you run

1. Colab Runtime -> Change runtime type -> **GPU (T4)**.
2. Colab Secrets / env values to set:
   - `WANDB_API_KEY`
   - `WANDB_ENTITY`
   - `HF_TOKEN` (needed only for model upload)
   - `SQLSAGE_ENV_URL` (your runtime URL, e.g. `https://adity00-sqlsage-env.hf.space`)
3. Run cells top-to-bottom. First goal is a stable smoke + baseline run.

## T4-safe defaults used below

- `max_seq_length=1024`
- `max_completion_length=256`
- `per_device_train_batch_size=2`
- `gradient_accumulation_steps=8`
- `num_generations=4`

After stability on T4, you can scale up.

In [ ]:
# Cell 1 — install dependencies and verify GPU

%pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
%pip install -q trl openenv-core wandb transformers datasets accelerate huggingface_hub requests psycopg2-binary

import os
import torch

assert torch.cuda.is_available(), "Switch Runtime -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

# Clone repo if not present (GitHub layout: pyproject.toml at repo root, not sqlsage-env/)
if not os.path.isdir("/content/SQLSage"):
    !git clone https://github.com/neelshingavi/SQLSage.git /content/SQLSage

%cd /content/SQLSage
import pathlib
assert pathlib.Path("pyproject.toml").is_file(), "Expected pyproject.toml at /content/SQLSage — pull latest main if missing"
%pip install -q -e '.[training,plots]'
print("Repo root:", pathlib.Path().resolve())

In [ ]:
# Cell 2 — set secrets, login, and validate Space endpoint

import os
import requests
import wandb

# Fill these if not already set in Colab environment/secrets
os.environ.setdefault("SQLSAGE_ENV_URL", "https://adity00-sqlsage-env.hf.space")
os.environ.setdefault("WANDB_ENTITY", "")
os.environ.setdefault("WANDB_PROJECT", "sqlsage-grpo")

base = os.environ["SQLSAGE_ENV_URL"].rstrip("/")
print("Using SQLSAGE_ENV_URL:", base)

# Login wandb (uses WANDB_API_KEY if present)
wandb.login()

# Endpoint smoke checks
health = requests.get(base + "/health", timeout=60)
print("/health status:", health.status_code, health.text)
health.raise_for_status()

reset = requests.post(base + "/reset", json={}, timeout=180)
print("/reset status:", reset.status_code)
reset.raise_for_status()

obs = reset.json().get("observation", {})
print("Initial task_level:", obs.get("task_level"), "step_count:", obs.get("step_count"))

In [ ]:
# Cell 3 — T4-safe rollout logging (smoke + baseline)

# This cell does not run GRPO yet. It verifies training plumbing and logs to wandb.
# You will get JSONL artifacts for baseline evidence and wandb curves immediately.

import os

base = os.environ["SQLSAGE_ENV_URL"].rstrip("/")
entity = os.environ.get("WANDB_ENTITY", "")
project = os.environ.get("WANDB_PROJECT", "sqlsage-grpo")

print("Running rollout_wandb.py against:", base)
print("WANDB_ENTITY:", entity if entity else "<default account>")
print("WANDB_PROJECT:", project)

# 5-episode smoke
!python scripts/rollout_wandb.py --episodes 5 --base-url "$SQLSAGE_ENV_URL" --project "$WANDB_PROJECT" --run-name "colab-smoke-5ep" --out-jsonl results/smoke.jsonl

# 50-episode baseline export (identity policy)
!python scripts/rollout_wandb.py --episodes 50 --policy identity --base-url "$SQLSAGE_ENV_URL" --project "$WANDB_PROJECT" --run-name "colab-baseline-50ep" --out-jsonl results/baseline.jsonl

!ls -lh results/smoke.jsonl results/baseline.jsonl

In [ ]:
# Cell 4 — load model (T4-safe) and print GRPO config

from unsloth import FastLanguageModel
from trl import GRPOConfig
from sqlsage.training.config import default_grpo_config

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=1024,   # T4-safe; raise later if stable
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

cfg_dict = default_grpo_config()
cfg_dict.update(
    {
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "num_generations": 4,
        "max_completion_length": 256,
    }
)
cfg = GRPOConfig(**cfg_dict)
print(cfg)

In [ ]:
# Cell 5 — Phase-8 outputs: plots, trained export placeholder, and upload commands

# 1) Regenerate judging plots from wandb data (falls back to synthetic if run not available yet)
!python plots/generate_plots.py

# 2) OPTIONAL: after your real trained-policy eval is wired, export to results/trained.jsonl
# !python scripts/rollout_wandb.py --episodes 50 --policy identity --base-url "$SQLSAGE_ENV_URL" --project "$WANDB_PROJECT" --run-name "colab-trained-50ep" --out-jsonl results/trained.jsonl

# 3) Build baseline-vs-trained table when trained.jsonl exists
# !python scripts/compare_rollouts.py --baseline results/baseline.jsonl --trained results/trained.jsonl --out results/baseline_vs_trained.md

# 4) Save merged model after GRPO training and upload to HF Hub
# model.save_pretrained_merged("sqlsage-trained", tokenizer, save_method="merged_16bit")
# import os
# os.environ["HF_TOKEN"] = "..."  # preferably set as secret
# !python scripts/push_model_to_hub.py --folder ./sqlsage-trained --repo-id YOUR_ORG/sqlsage-trained

!ls -lh plots/reward_curve.png plots/penalty_dashboard.png plots/plan_improvement.png